# Natural gas forecasting: GRU + Multi-Head Attention

One-step-ahead forecasting of daily natural gas power generation (GWh) in Mexico with a GRU layer returning the full sequence, multi-head self-attention, layer normalisation, global average pooling, dropout and a linear output.

Part of the experiments for *Forecasting Natural Gas for Power Generation Using Attention-Based Deep Learning Model* (Springer, 2026). The shared pipeline lives in [`src/forecasting.py`](../src/forecasting.py).

## Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # make `src` importable

from src import forecasting as fc

MODEL = "gru_attention"
fc.set_seed()

## Data

Daily generation from 1 January 2022 to 30 April 2025.

In [ ]:
data = fc.load_data()
fc.plot_series(data)
data.describe()

## Preprocessing

Sliding windows of the previous `LOOKBACK` = 30 days predict the next day. Windows are split chronologically into 70% train, 15% validation and 15% test. The min-max scaler is fitted on the training period only.

In [ ]:
ds = fc.prepare_dataset(data)
for split in ("train", "val", "test"):
    dates = dict(zip(("train", "val", "test"), ds.split_dates))[split]
    print(f"{split:>5}: {len(dates):4d} windows, "
          f"{dates.iloc[0]:%Y-%m-%d} to {dates.iloc[-1]:%Y-%m-%d}")

## Hyperparameter tuning

Random search (`MAX_TRIALS` trials, early stopping) selected on validation loss.

In [ ]:
best_hp = fc.tune(MODEL, ds)
best_hp.values

## Training

A fresh model is trained with the selected hyperparameters.

In [ ]:
model, history = fc.train(MODEL, best_hp, ds)
model.summary()
fc.plot_learning_curve(history)

## Evaluation

Metrics are in original units (GWh). **The test split is only used here**, never for tuning or early stopping.

In [ ]:
preds = fc.predict_splits(model, ds)
fc.evaluate(MODEL, best_hp, preds)

## 90-day forecast

Recursive forecast: each prediction is fed back as the newest input.

In [ ]:
forecast = fc.forecast(model, ds)
fc.plot_predictions(MODEL, ds, preds, forecast)
forecast.head(10)